# 03 · Chat API with FastAPI

The "wrap an LLM in a service" round. You will build a chat backend from a blank file, piece by piece: pydantic request and response models, `POST /chat` with a conversation id and history, `POST /chat/stream` over Server-Sent Events, input validation, a per-user rate limit, provider errors mapped to honest HTTP codes, and a test suite that runs with `TestClient`. Then you run it on a real server and measure time-to-first-token.

**Time:** ~35 min · **Runs:** offline on CPU in < 1 min · **Needs:** [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb) goes deeper. [Async basics](../01_python_basics/14_async_await_basics.ipynb) helps.

## Why this matters in interviews

- Almost every GenAI team ships models *behind an API*. "Build me the chat endpoint" tests whether you can write production-shaped Python: typed models, dependencies, error handling and tests.
- Streaming is where LLM services differ from normal web services. There is time-to-first-token, and errors can happen *after* the `200` has been sent.
- The discussion afterwards (async clients, timeouts, retries, where state lives, horizontal scaling) is a system-design round in miniature.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl07` | Compare FastAPI and Streamlit. |
| `tl08` | Why would you use async for an endpoint that calls an LLM? |
| `pd12` | How do you rate limit an LLM API you own? |
| `po07` | You are getting 429s from your model provider — what is the handling? |
| `po17` | Why are timeouts more delicate in an LLM system than in a normal service? |
| `pd39` | What does "streaming" mean in a production LLM system, and what are the gotchas? |
| `in08` | What is streaming, and what does it actually improve? |
| `po05` | What is horizontal scaling and where does it stop helping a GenAI system? |
| `pd29` | How do you test an LLM feature in CI without spending a fortune? |
| `sf20` | What is data leakage between users, and how does it happen? |

## The question, as the interviewer asks it

> *"Build a small chat service in front of an LLM. A client sends a message and optionally a conversation id, and gets a reply. The service remembers the conversation, can stream the reply token by token, validates input, limits each user's request rate, and doesn't leak raw provider errors. Show me that it works with tests."*

## The first 60 seconds

| Clarify | Why |
|---|---|
| Who is the user, and how do we identify them? | Rate limits and history are **per user**. Here an `X-User-Id` header stands in for what your auth gateway would set. |
| Streaming: SSE or WebSockets? | SSE is one-directional, plain HTTP and proxy-friendly, which is all a chat reply needs. WebSockets are for two-way, real-time traffic. |
| Where does conversation state live? | In memory for this round, **Redis or Postgres in production**, because more than one worker must see it. |
| Limits: message length, history length, requests per minute? | These become pydantic constraints, a trimming policy and a token bucket. |

**The plan, out loud:** "Pydantic models for the request and response. A conversation store keyed by *(user, conversation id)* so users can't read each other's history. A token-bucket rate limiter as a FastAPI dependency. `POST /chat` returns JSON, `POST /chat/stream` returns `text/event-stream`. Exception handlers map OpenAI errors to 429, 502 and 504. The LLM client comes in as a dependency, so the tests can swap in a fake. Then the tests."

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER
from llm_setup import make_async_client, OFFLINE

## Packages

```
pip install fastapi uvicorn openai httpx pydantic
```

| Package | What it gives you | Swap it for |
|---|---|---|
| `fastapi` | routing, dependency injection, request validation from type hints, OpenAPI docs at `/docs` | Litestar, Flask plus pydantic (sync), Django Ninja |
| `pydantic` | the request and response models, with constraints (`min_length`, `pattern`) that become 422 errors | attrs plus manual checks |
| `openai` | `AsyncOpenAI`, so one worker can wait on many LLM calls at once | `litellm` (many providers behind one API), `anthropic` |
| `uvicorn` | the ASGI server that actually listens on a port | Hypercorn, Granian, and `gunicorn -k uvicorn.workers.UvicornWorker` for several processes |
| `httpx` | `TestClient` is built on it, and it is also a streaming HTTP client | `requests` (no async, no HTTP/2) |
| *(in production)* `redis` | shared conversation state and shared rate-limit counters | Postgres for durable transcripts; `slowapi` or the API gateway for rate limiting; `sse-starlette` for SSE |

## Step 1 · Request and response models

Validation lives in the types. FastAPI turns any violation into a `422` with a precise error body, before your code runs.

In [ ]:
import json
import math
import time
import uuid

import openai
from fastapi import Depends, FastAPI, Header, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse
from pydantic import BaseModel, Field

class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=2000, description="the user's message")
    conversation_id: str | None = Field(default=None, pattern=r"^[A-Za-z0-9_-]{1,64}$",
                                        description="omit to start a new conversation")

class ChatResponse(BaseModel):
    conversation_id: str
    reply: str
    turns: int

print(ChatRequest(message="hi").model_dump())
try:
    ChatRequest(message="", conversation_id="../etc/passwd")
except Exception as err:
    print(type(err).__name__, "->", [e["loc"][0] + ": " + e["msg"] for e in err.errors()])

## Step 2 · The conversation store and the rate limiter

**Store.** A dict keyed by `(user_id, conversation_id)`. Putting the user in the key is the cheapest defence against one user reading another's conversation by guessing an id (`sf20`). History is trimmed to the last N messages so the prompt cannot grow without bound. In production this is Redis (a list per key with a TTL) behind the same three methods.

**Rate limiter.** A token bucket per user. Each user has `capacity` tokens, every request spends one, and tokens refill at `rate` per second. Bursts up to `capacity` are allowed, and the long-run rate is capped. The clock is injectable so the tests are deterministic.

In [ ]:
class ConversationStore:
    def __init__(self, max_messages=20):
        self.max_messages, self._data = max_messages, {}
    def history(self, user, cid):
        return list(self._data.get((user, cid), []))
    def append(self, user, cid, *messages):
        h = self._data.setdefault((user, cid), [])
        h.extend(messages)
        del h[:-self.max_messages]                  # keep the newest N (even N keeps user/assistant pairs)
    def turns(self, user, cid):
        return len(self._data.get((user, cid), [])) // 2

class TokenBucket:
    def __init__(self, capacity=5, rate=1.0, clock=time.monotonic):
        self.capacity, self.rate, self.clock, self._state = capacity, rate, clock, {}
    def allow(self, key):
        """Returns (allowed, seconds until the next token)."""
        now = self.clock()
        tokens, last = self._state.get(key, (self.capacity, now))
        tokens = min(self.capacity, tokens + (now - last) * self.rate)
        if tokens >= 1:
            self._state[key] = (tokens - 1, now)
            return True, 0.0
        self._state[key] = (tokens, now)
        return False, (1 - tokens) / self.rate

class FakeClock:                                    # for tests and the chart below
    def __init__(self): self.t = 0.0
    def __call__(self): return self.t

clock = FakeClock()
bucket = TokenBucket(capacity=3, rate=0.5, clock=clock)
print([bucket.allow("u1")[0] for _ in range(4)], "<- burst of 4: capacity 3")
clock.t += 2.0
print(bucket.allow("u1"), "<- 2 s later one token has refilled")
print(bucket.allow("u2"), "<- another user has their own bucket")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(0)
clock = FakeClock()
bucket = TokenBucket(capacity=5, rate=1.0, clock=clock)
arrivals = np.sort(np.concatenate([rng.uniform(0, 1.0, 12), rng.uniform(4, 10, 8)]))   # a burst, then a trickle
ok = []
for t in arrivals:
    clock.t = t
    ok.append(bucket.allow("u1")[0])
ok = np.array(ok)
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.scatter(arrivals[ok], np.ones(ok.sum()), marker="o", s=60, color="#2f9e44", label="allowed")
ax.scatter(arrivals[~ok], np.ones((~ok).sum()), marker="x", s=60, color="#e03131", label="429")
ax.set_yticks([]); ax.set_xlabel("seconds"); ax.legend(loc="upper right")
ax.set_title("Token bucket (capacity 5, 1 token/s): a burst of 12 requests, 5 pass; the later trickle all passes")
plt.tight_layout(); plt.show()
print(f"burst: {ok[arrivals < 1].sum()} of {(arrivals < 1).sum()} allowed; trickle: {ok[arrivals >= 4].sum()} of {(arrivals >= 4).sum()}")
assert ok[arrivals < 1].sum() == 5 and ok[arrivals >= 4].all()

## Step 3 · The app, its dependencies and the error mapping

Three dependencies do the cross-cutting work, and each one can be overridden in tests:

- `get_llm()` returns the shared `AsyncOpenAI` client. Create it **once**, since it holds a connection pool, and give it an explicit timeout and a retry count.
- `get_store()` returns the conversation store.
- `current_user()` reads `X-User-Id`, applies the rate limit, and raises `429` with a `Retry-After` header.

Exception handlers turn provider failures into status codes a client can act on. Note that FastAPI picks the handler by the exception's class hierarchy, so the most specific one wins.

| Upstream failure | We return | Why |
|---|---|---|
| `openai.RateLimitError` (provider 429) | **503** + `Retry-After` | It is *our* dependency that is overloaded, not the client misbehaving. Some teams pass 429 through. Either way, say which and why |
| `openai.APITimeoutError` | **504** Gateway Timeout | the upstream did not answer in time |
| `openai.APIConnectionError` | **502** Bad Gateway | we could not reach the upstream |
| any other `openai.APIStatusError` | **502**, logging the upstream status and request id | never forward the provider's raw error body to users |

In [ ]:
SYSTEM = "You are a concise, friendly assistant for an online shop."
llm_client = make_async_client(timeout=30.0, max_retries=2)   # app.py: AsyncOpenAI(timeout=30.0, max_retries=2)
store = ConversationStore(max_messages=20)
limiter = TokenBucket(capacity=5, rate=1.0)

app = FastAPI(title="chat-service")

def get_llm():
    return llm_client

def get_store():
    return store

def current_user(x_user_id: str = Header(min_length=1, max_length=64)):
    allowed, retry_after = limiter.allow(x_user_id)
    if not allowed:
        raise HTTPException(429, detail="rate limit exceeded", headers={"Retry-After": str(math.ceil(retry_after))})
    return x_user_id

def error(status, code, message, **headers):
    return JSONResponse(status_code=status, content={"error": code, "message": message}, headers=headers or None)

@app.exception_handler(openai.RateLimitError)
async def upstream_rate_limited(request: Request, exc: openai.RateLimitError):
    return error(503, "upstream_rate_limited", "The model provider is busy. Please retry shortly.", **{"Retry-After": "5"})

@app.exception_handler(openai.APITimeoutError)
async def upstream_timeout(request: Request, exc: openai.APITimeoutError):
    return error(504, "upstream_timeout", "The model took too long to answer.")

@app.exception_handler(openai.APIConnectionError)
async def upstream_unreachable(request: Request, exc: openai.APIConnectionError):
    return error(502, "upstream_unreachable", "Could not reach the model provider.")

@app.exception_handler(openai.APIStatusError)
async def upstream_error(request: Request, exc: openai.APIStatusError):
    print(f"upstream {exc.status_code} request_id={exc.request_id}")    # a log line, not the user's problem
    return error(502, "upstream_error", "The model provider returned an error.")

print([r.path for r in app.routes])

## Step 4 · `POST /chat` and `POST /chat/stream`

`/chat` is plain request/response. `/chat/stream` sends Server-Sent Events: each event is a `data: <json>\n\n` block. A `done` event carries the conversation id, and an `error` event is sent if the provider fails **mid-stream**. That last case is the streaming gotcha: once the first byte has gone out, the status is already `200` and cannot change. Errors *before* the stream starts still go through the exception handlers, because the `create(..., stream=True)` call is awaited before the `StreamingResponse` is returned.

In [ ]:
def build_messages(history, text):
    return [{"role": "system", "content": SYSTEM}, *history, {"role": "user", "content": text}]

@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest, user: str = Depends(current_user),
               llm=Depends(get_llm), store: ConversationStore = Depends(get_store)):
    cid = req.conversation_id or uuid.uuid4().hex[:12]
    resp = await llm.chat.completions.create(model=MODEL, messages=build_messages(store.history(user, cid), req.message),
                                             temperature=0.3, max_tokens=300)
    reply = resp.choices[0].message.content
    store.append(user, cid, {"role": "user", "content": req.message}, {"role": "assistant", "content": reply})
    return ChatResponse(conversation_id=cid, reply=reply, turns=store.turns(user, cid))

def sse(data, event=None):
    return (f"event: {event}\n" if event else "") + f"data: {json.dumps(data)}\n\n"

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest, user: str = Depends(current_user),
                      llm=Depends(get_llm), store: ConversationStore = Depends(get_store)):
    cid = req.conversation_id or uuid.uuid4().hex[:12]
    stream = await llm.chat.completions.create(model=MODEL, messages=build_messages(store.history(user, cid), req.message),
                                               temperature=0.3, max_tokens=300, stream=True)
    async def events():
        parts = []
        try:
            async for chunk in stream:
                delta = chunk.choices[0].delta.content if chunk.choices else None
                if delta:
                    parts.append(delta)
                    yield sse({"delta": delta})
        except openai.APIError as err:                # after the 200: report in-band
            yield sse({"error": type(err).__name__}, event="error")
            return
        store.append(user, cid, {"role": "user", "content": req.message},
                     {"role": "assistant", "content": "".join(parts)})
        yield sse({"conversation_id": cid, "turns": store.turns(user, cid)}, event="done")
    return StreamingResponse(events(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

@app.get("/healthz")
async def healthz():
    return {"ok": True}

print([f"{sorted(r.methods)[0]} {r.path}" for r in app.routes if hasattr(r, "methods")][-3:])

## Step 5 · Try it with `TestClient`

`TestClient` runs the app in-process. There is no server and no port, and every request goes through the full FastAPI stack: validation, dependencies, handlers. With the offline stand-in, the "remember my name" exchange works because the history really is sent back to the model on turn two.

In [ ]:
from fastapi.testclient import TestClient

api = TestClient(app)
H = {"X-User-Id": "alice"}

r1 = api.post("/chat", json={"message": "Hi! My name is Sujan."}, headers=H)
cid = r1.json()["conversation_id"]
r2 = api.post("/chat", json={"message": "What is my name?", "conversation_id": cid}, headers=H)
print(r1.status_code, r1.json()["reply"][:70])
print(r2.status_code, r2.json())
assert r2.json()["turns"] == 2 and "Sujan" in r2.json()["reply"]

r3 = api.post("/chat", json={"message": "What is my name?", "conversation_id": cid}, headers={"X-User-Id": "mallory"})
print("same conversation id, different user ->", r3.json()["reply"])
assert "Sujan" not in r3.json()["reply"]

In [ ]:
with api.stream("POST", "/chat/stream", json={"message": "Summarize: Our store ships worldwide. Returns are free.",
                                              "conversation_id": cid}, headers=H) as resp:
    print("status:", resp.status_code, "| content-type:", resp.headers["content-type"])
    raw = "".join(resp.iter_text())

events = [block for block in raw.strip().split("\n\n")]
print(f"{len(events)} SSE events. First three and the last:")
for block in events[:3] + events[-1:]:
    print("   ", block.replace("\n", " | "))
text = "".join(json.loads(b[len("data: "):])["delta"] for b in events if b.startswith("data: "))
print("reassembled:", text)
assert events[-1].startswith("event: done")

The wire format is the whole trick: `data: {...}` blocks separated by a blank line, with an optional `event:` line. A browser reads it with `EventSource` (GET only) or `fetch()` plus a stream reader (works with POST). Recent FastAPI versions, including the 0.141 in this venv, also ship `fastapi.sse.EventSourceResponse` and `ServerSentEvent`. With those you `yield` dicts and FastAPI does the encoding. The hand-written version above works on every version, and it shows what goes over the wire.

## Step 6 · The test suite

Plain `test_*` functions using `assert`, the way pytest would collect them. The error-mapping tests use **`app.dependency_overrides`** to swap the LLM for a fake that raises. This is how you test failure paths in CI without a network, a key or any cost (`pd29`).

In [ ]:
import httpx
from types import SimpleNamespace

_req = httpx.Request("POST", "https://api.openai.com/v1/chat/completions")

class FailingLLM:
    """Looks like AsyncOpenAI (llm.chat.completions.create) but raises the error we ask for."""
    def __init__(self, exc):
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
        self.exc = exc
    async def _create(self, **kwargs):
        raise self.exc

def fresh_client(llm=None):
    """Each test gets a clean store and limiter; pass llm= to replace the model with a fake."""
    app.dependency_overrides.clear()
    new_store = ConversationStore()
    app.dependency_overrides[get_store] = lambda: new_store
    if llm is not None:
        app.dependency_overrides[get_llm] = lambda: llm
    limiter._state.clear()
    limiter.clock = FakeClock()                  # frozen time: no refills mid-test, so results are exact
    return TestClient(app)

def test_chat_happy_path():
    r = fresh_client().post("/chat", json={"message": "What is 6 * 7?"}, headers=H)
    assert r.status_code == 200 and r.json()["turns"] == 1 and "42" in r.json()["reply"]

def test_validation_errors_are_422():
    c = fresh_client()
    assert c.post("/chat", json={"message": ""}, headers=H).status_code == 422
    assert c.post("/chat", json={"message": "x" * 2001}, headers=H).status_code == 422
    assert c.post("/chat", json={"message": "hi", "conversation_id": "a/b"}, headers=H).status_code == 422
    assert c.post("/chat", json={"message": "hi"}).status_code == 422            # no X-User-Id header

def test_rate_limit_is_per_user():
    c = fresh_client()
    codes = [c.post("/chat", json={"message": "hi"}, headers=H).status_code for _ in range(7)]
    assert codes[:5] == [200] * 5 and codes[5:] == [429, 429], codes
    r = c.post("/chat", json={"message": "hi"}, headers=H)
    assert int(r.headers["Retry-After"]) >= 1
    assert c.post("/chat", json={"message": "hi"}, headers={"X-User-Id": "bob"}).status_code == 200

def test_provider_429_maps_to_503():
    exc = openai.RateLimitError("slow down", response=httpx.Response(429, request=_req), body=None)
    r = fresh_client(llm=FailingLLM(exc)).post("/chat", json={"message": "hi"}, headers=H)
    assert r.status_code == 503 and r.json()["error"] == "upstream_rate_limited" and r.headers["Retry-After"] == "5"

def test_provider_timeout_maps_to_504():
    r = fresh_client(llm=FailingLLM(openai.APITimeoutError(request=_req))).post(
        "/chat", json={"message": "hi"}, headers=H)
    assert r.status_code == 504 and r.json()["error"] == "upstream_timeout"

def test_provider_500_maps_to_502_without_leaking():
    exc = openai.InternalServerError("secret upstream detail", response=httpx.Response(500, request=_req), body=None)
    r = fresh_client(llm=FailingLLM(exc)).post("/chat", json={"message": "hi"}, headers=H)
    assert r.status_code == 502 and "secret" not in r.text

def test_stream_ends_with_done_event():
    with fresh_client().stream("POST", "/chat/stream", json={"message": "hello"}, headers=H) as r:
        body = "".join(r.iter_text())
    assert r.status_code == 200 and body.rstrip().split("\n\n")[-1].startswith("event: done")

TESTS = [f for name, f in list(globals().items()) if name.startswith("test_") and callable(f)]
for t in TESTS:
    t()
    print(f"PASS  {t.__name__}")
app.dependency_overrides.clear()
print(f"\n{len(TESTS)} passed")

The "upstream 500" line printed above comes from the handler's log statement, not from the response. The client only saw a generic `upstream_error`, which is what `test_provider_500_maps_to_502_without_leaking` checks.

## Step 7 · Run it on a real server and measure time-to-first-token

`TestClient` collects the whole streamed body before handing it back, so it cannot show *when* the tokens arrive. Start uvicorn in a background thread and use `httpx` as a real client. Offline, the stand-in waits `token_delay` seconds between streamed chunks, much like a real model generating tokens.

In [ ]:
import socket
import threading
import uvicorn

if OFFLINE:
    OFFLINE.token_delay = 0.03                   # the stand-in "generates" a chunk every 30 ms
limiter.capacity, limiter.clock = 100, time.monotonic   # real clock again; don't rate-limit the benchmark

with socket.socket() as s:                       # a free port on localhost
    s.bind(("127.0.0.1", 0)); PORT = s.getsockname()[1]
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=PORT, log_level="warning"))
thread = threading.Thread(target=server.run, daemon=True)
thread.start()
while not server.started:
    time.sleep(0.05)
BASE = f"http://127.0.0.1:{PORT}"
print("serving on", BASE, "->", httpx.get(BASE + "/healthz").json())

In [ ]:
PROMPT = {"message": "Summarize: Orders ship in two days. Returns are free for 30 days. Support is open 24/7. "
                     "Gift cards never expire. Express delivery costs extra."}
first_byte, complete, n_events = [], [], []
with httpx.Client(base_url=BASE, headers=H, timeout=30) as http:
    for _ in range(5):
        t0, first, events = time.perf_counter(), None, 0
        with http.stream("POST", "/chat/stream", json=PROMPT) as r:
            for line in r.iter_lines():
                if line.startswith("data:"):
                    events += 1
                    if first is None:
                        first = time.perf_counter() - t0
        first_byte.append(first); complete.append(time.perf_counter() - t0); n_events.append(events)
server.should_exit = True
thread.join(timeout=5)
if OFFLINE:
    OFFLINE.token_delay = 0.0

ttft, total = np.median(first_byte) * 1000, np.median(complete) * 1000
print(f"median of 5 runs, {n_events[0]} SSE events each: first token {ttft:.0f} ms, complete {total:.0f} ms")
assert ttft < total / 3

fig, ax = plt.subplots(figsize=(9, 2.8))
runs = np.arange(1, 6)
ax.barh(runs, np.array(complete) * 1000, color="#dee2e6", label="stream complete")
ax.barh(runs, np.array(first_byte) * 1000, color="#4c6ef5", label="first token arrives")
ax.set_yticks(runs, [f"run {i}" for i in runs]); ax.set_xlabel("milliseconds")
ax.set_title("POST /chat/stream on a real server: the first word arrives long before the answer is done")
ax.legend(loc="lower right"); plt.tight_layout(); plt.show()

That chart is the whole argument for streaming (`in08`). Streaming does **not** make the answer finish sooner, because the model generates at the same speed either way. A non-streaming `/chat` shows the user *nothing* until the grey bar ends. With streaming, the first words appear at the blue bar, and time to first token is what users perceive as speed. Offline the numbers come from the stand-in's 30 ms per chunk. (It adds that delay only when streaming, so timing `/chat` offline would look instant and prove nothing.) With a real model, the first token typically arrives in a few hundred milliseconds and a full answer takes several seconds, so the gap is larger.

## The whole thing as one file: `app.py`

Paste this into `app.py` and run `uvicorn app:app --reload`. It needs `pip install fastapi uvicorn openai` and `OPENAI_API_KEY`. Interactive docs appear at `http://127.0.0.1:8000/docs`.

```python
"""app.py - a chat service in front of an LLM.
pip install fastapi uvicorn openai ; env: OPENAI_API_KEY ; run: uvicorn app:app --reload"""
import json
import math
import time
import uuid

import openai
from fastapi import Depends, FastAPI, Header, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

MODEL = "gpt-4.1-mini"
SYSTEM = "You are a concise, friendly assistant for an online shop."


# ---- models: validation lives in the types (violations -> 422)
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=2000)
    conversation_id: str | None = Field(default=None, pattern=r"^[A-Za-z0-9_-]{1,64}$")

class ChatResponse(BaseModel):
    conversation_id: str
    reply: str
    turns: int


# ---- state: in memory here; Redis (history + counters) once there is more than one worker
class ConversationStore:
    def __init__(self, max_messages=20):
        self.max_messages, self._data = max_messages, {}
    def history(self, user, cid):
        return list(self._data.get((user, cid), []))
    def append(self, user, cid, *messages):
        h = self._data.setdefault((user, cid), [])
        h.extend(messages)
        del h[:-self.max_messages]
    def turns(self, user, cid):
        return len(self._data.get((user, cid), [])) // 2

class TokenBucket:
    def __init__(self, capacity=5, rate=1.0, clock=time.monotonic):
        self.capacity, self.rate, self.clock, self._state = capacity, rate, clock, {}
    def allow(self, key):
        now = self.clock()
        tokens, last = self._state.get(key, (self.capacity, now))
        tokens = min(self.capacity, tokens + (now - last) * self.rate)
        if tokens >= 1:
            self._state[key] = (tokens - 1, now)
            return True, 0.0
        self._state[key] = (tokens, now)
        return False, (1 - tokens) / self.rate


llm_client = AsyncOpenAI(timeout=30.0, max_retries=2)   # one shared client: it owns the connection pool
store = ConversationStore()
limiter = TokenBucket(capacity=5, rate=1.0)
app = FastAPI(title="chat-service")


# ---- dependencies: each one can be overridden in tests
def get_llm():
    return llm_client

def get_store():
    return store

def current_user(x_user_id: str = Header(min_length=1, max_length=64)):   # set by your auth gateway
    allowed, retry_after = limiter.allow(x_user_id)
    if not allowed:
        raise HTTPException(429, detail="rate limit exceeded", headers={"Retry-After": str(math.ceil(retry_after))})
    return x_user_id


# ---- provider errors -> honest status codes, no raw upstream bodies
def error(status, code, message, **headers):
    return JSONResponse(status_code=status, content={"error": code, "message": message}, headers=headers or None)

@app.exception_handler(openai.RateLimitError)
async def upstream_rate_limited(request: Request, exc: openai.RateLimitError):
    return error(503, "upstream_rate_limited", "The model provider is busy. Please retry shortly.", **{"Retry-After": "5"})

@app.exception_handler(openai.APITimeoutError)
async def upstream_timeout(request: Request, exc: openai.APITimeoutError):
    return error(504, "upstream_timeout", "The model took too long to answer.")

@app.exception_handler(openai.APIConnectionError)
async def upstream_unreachable(request: Request, exc: openai.APIConnectionError):
    return error(502, "upstream_unreachable", "Could not reach the model provider.")

@app.exception_handler(openai.APIStatusError)
async def upstream_error(request: Request, exc: openai.APIStatusError):
    print(f"upstream {exc.status_code} request_id={exc.request_id}")
    return error(502, "upstream_error", "The model provider returned an error.")


# ---- routes
def build_messages(history, text):
    return [{"role": "system", "content": SYSTEM}, *history, {"role": "user", "content": text}]

@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest, user: str = Depends(current_user),
               llm=Depends(get_llm), store: ConversationStore = Depends(get_store)):
    cid = req.conversation_id or uuid.uuid4().hex[:12]
    resp = await llm.chat.completions.create(model=MODEL, messages=build_messages(store.history(user, cid), req.message),
                                             temperature=0.3, max_tokens=300)
    reply = resp.choices[0].message.content
    store.append(user, cid, {"role": "user", "content": req.message}, {"role": "assistant", "content": reply})
    return ChatResponse(conversation_id=cid, reply=reply, turns=store.turns(user, cid))

def sse(data, event=None):
    return (f"event: {event}\n" if event else "") + f"data: {json.dumps(data)}\n\n"

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest, user: str = Depends(current_user),
                      llm=Depends(get_llm), store: ConversationStore = Depends(get_store)):
    cid = req.conversation_id or uuid.uuid4().hex[:12]
    stream = await llm.chat.completions.create(model=MODEL, messages=build_messages(store.history(user, cid), req.message),
                                               temperature=0.3, max_tokens=300, stream=True)
    async def events():
        parts = []
        try:
            async for chunk in stream:
                delta = chunk.choices[0].delta.content if chunk.choices else None
                if delta:
                    parts.append(delta)
                    yield sse({"delta": delta})
        except openai.APIError as err:            # the 200 is already sent: report in-band
            yield sse({"error": type(err).__name__}, event="error")
            return
        store.append(user, cid, {"role": "user", "content": req.message},
                     {"role": "assistant", "content": "".join(parts)})
        yield sse({"conversation_id": cid, "turns": store.turns(user, cid)}, event="done")
    return StreamingResponse(events(), media_type="text/event-stream",
                             headers={"Cache-Control": "no-cache", "X-Accel-Buffering": "no"})

@app.get("/healthz")
async def healthz():
    return {"ok": True}
```

## How to explain the key lines

| Line | What to say |
|---|---|
| `Field(min_length=1, max_length=2000)`, `pattern=...` | "Validation is declarative. Bad input is a 422 before my code runs, and the limits also cap prompt cost." |
| `AsyncOpenAI(timeout=30.0, max_retries=2)` | "One shared async client. While one request waits on the model, the worker serves others. The timeout and retry count are explicit, because the defaults (10 minutes, 2 retries) are wrong for a user-facing path." |
| `Depends(current_user)` | "Auth and rate limiting as a dependency: reusable, testable, overridable." |
| key `(user, cid)` | "History is scoped to the user, so guessing someone's conversation id gets you nothing." |
| `del h[:-self.max_messages]` | "A bounded context. Long chats don't grow the prompt, and the cost, forever." |
| `@app.exception_handler(openai.RateLimitError)` | "Provider failures become statuses clients can act on, with `Retry-After`. The raw upstream error only goes to the log." |
| `StreamingResponse(..., media_type="text/event-stream")` | "SSE: `data:` blocks over one HTTP response. `X-Accel-Buffering: no` stops nginx from buffering the stream." |
| `yield sse(..., event="error")` | "Mid-stream failures can't change the status code, so they are reported in the stream." |
| `app.dependency_overrides[get_llm]` | "Tests swap the model for a fake. Failure paths get tested in CI with no network and no cost." |

## The follow-ups they will ask

| Ask | Answer |
|---|---|
| **Why async?** (`tl08`) | An LLM call is 1–10 s of *waiting*. With `async def` and `AsyncOpenAI`, one worker process holds hundreds of in-flight calls. A sync client inside `async def` blocks the event loop, which is the classic bug. |
| **Timeouts?** (`po17`) | Set them per call and match them to the use case. A 60 s generation is normal, a 60 s hang is not. Use a short connect timeout, a longer read timeout, and for streams a *per-chunk* idle timeout. Your server's timeout must exceed the upstream's, or you cancel work you are paying for. |
| **Retries?** (`po16`) | Retry 429, 5xx and connection errors with exponential backoff and jitter, 2–3 attempts, and **not** on 400s. Never retry a stream that has already sent tokens to the user. The SDK's `max_retries` covers the basics. |
| **Observability?** | A request id per call (pass the provider's `x-request-id` into your logs), latency and TTFT histograms, tokens and cost per user, the error rate by type. OpenTelemetry spans around the model call. |
| **Horizontal scaling?** (`po05`) | Workers must be **stateless**, so history and rate-limit counters move to Redis. Then add processes and machines behind a load balancer. Scaling stops helping when the *provider's* rate limit is the bottleneck. After that it is quotas, queues, caching and model routing. |
| **Where should state live?** | Conversation history in Redis with a TTL (fast, shared) plus Postgres for durable transcripts and audit. Rate-limit counters in Redis or at the API gateway. Never in process memory once there are two workers. |
| **Client disconnects mid-stream?** | Starlette cancels the generator. Wrap the provider stream so the upstream request is closed too, or you keep paying for tokens nobody reads. |

## Try it yourself

**1.** Add `GET /conversations/{conversation_id}` that returns the stored history for the calling user, or 404 if there is none. Test that alice can read hers and mallory gets a 404 for the same id.

In [ ]:
# Solution — try it yourself first, then run this
@app.get("/conversations/{conversation_id}")
async def get_conversation(conversation_id: str, user: str = Depends(current_user),
                           store: ConversationStore = Depends(get_store)):
    history = store.history(user, conversation_id)
    if not history:
        raise HTTPException(404, detail="no such conversation")
    return {"conversation_id": conversation_id, "messages": history}

c = fresh_client()
cid = c.post("/chat", json={"message": "Hello there"}, headers=H).json()["conversation_id"]
mine = c.get(f"/conversations/{cid}", headers=H)
theirs = c.get(f"/conversations/{cid}", headers={"X-User-Id": "mallory"})
print(mine.status_code, [m["role"] for m in mine.json()["messages"]], "| mallory:", theirs.status_code)
assert mine.status_code == 200 and theirs.status_code == 404
app.dependency_overrides.clear()

**2.** The stream endpoint reports provider errors that happen *after* the first byte. Write a test with a fake LLM whose stream yields one chunk and then raises `openai.APIConnectionError`. Assert the response is `200` and ends with an `event: error` block.

In [ ]:
# Solution — try it yourself first, then run this
class HalfStreamLLM:
    def __init__(self):
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
    async def _create(self, **kwargs):
        async def gen():
            yield SimpleNamespace(choices=[SimpleNamespace(delta=SimpleNamespace(content="Hel"))])
            raise openai.APIConnectionError(request=_req)
        return gen()

with fresh_client(llm=HalfStreamLLM()).stream("POST", "/chat/stream", json={"message": "hi"},
                                                               headers=H) as r:
    body = "".join(r.iter_text())
print(r.status_code, body.strip().split("\n\n"))
assert r.status_code == 200 and body.strip().split("\n\n")[-1].startswith("event: error")
app.dependency_overrides.clear()

## Say it in an interview

**30 seconds:** "FastAPI with pydantic models, so bad input is a 422 before my code runs. One shared `AsyncOpenAI` client with an explicit timeout and retries. History is stored per user and conversation id and trimmed, in memory here and in Redis in production. The rate limit is a token bucket per user, applied as a dependency that returns 429 with `Retry-After`. Provider errors are mapped to 503, 504 or 502 without leaking upstream details. Streaming is SSE, which puts errors in-band once the 200 is out. Everything is tested with `TestClient`, and dependency overrides fake the model so failure paths run in CI for free."

**Numbers:** users notice time-to-first-token. A few hundred ms feels instant, and several seconds of blank screen feels broken. The SDK default is 2 retries. A token bucket with capacity C and rate R allows bursts of C and R requests/s sustained.

**Traps:** a sync OpenAI client inside `async def` blocks the loop. Global in-memory state breaks with more than one worker. Returning the provider's error body leaks internals. A stream can't change its status code after the first byte. `TestClient` buffers streams, so measure TTFT against a real server. Proxies buffer SSE unless told not to. A missing user scope on conversation ids leaks data between users.

## Next

- Deeper: [FastAPI serving and streaming](../16_production/04_fastapi_serving_and_streaming.ipynb) · [retries, backoff, rate limits, circuit breakers](../16_production/02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [async concurrency for LLM calls](../16_production/01_async_concurrency_for_llm_calls.ipynb) · [latency percentiles and cost](../16_production/05_latency_percentiles_p50_p95_and_cost.ipynb) · [observability](../16_production/08_observability_logging_and_tracing.ipynb)
- Other rounds: [build an agent in 30 minutes](02_build_an_agent_in_30_minutes.ipynb) · [semantic search service](04_semantic_search_service.ipynb)
- Theory: [ai_system_design_concepts course](../../ai_system_design_concepts/index.html) · [ai_tooling course](../../ai_tooling/index.html) · [interview bank](../../ai_interview_prep/index.html)